# 04 - Remaining Categorical Encoding

Handles the remaining categorical fields not already addressed by target encoding: one hot encoding for low cardinality fields (`Shipping Mode`, `Customer Segment`, `Type`), and a frequency based encoding for `Category Name` (moderate cardinality, ~50 categories, confirmed reliable after the department scoping resolution in Stage 3).


## Setup

In [1]:
import pandas as pd
from pathlib import Path

TRAIN_IN = Path('../../../data/processed/features_step3_train.csv')
VAL_IN = Path('../../../data/processed/features_step3_val.csv')
TEST_IN = Path('../../../data/processed/features_step3_test.csv')

TRAIN_OUT = Path('../../../data/processed/features_step4_train.csv')
VAL_OUT = Path('../../../data/processed/features_step4_val.csv')
TEST_OUT = Path('../../../data/processed/features_step4_test.csv')

train_df = pd.read_csv(TRAIN_IN)
val_df = pd.read_csv(VAL_IN)
test_df = pd.read_csv(TEST_IN)

for col in ['Shipping Mode', 'Customer Segment', 'Type']:
    print(f"{col}: {train_df[col].nunique()} unique values")


Shipping Mode: 4 unique values
Customer Segment: 3 unique values
Type: 4 unique values


## 1. One hot encode low cardinality categoricals

In [2]:
onehot_cols = ['Shipping Mode', 'Customer Segment', 'Type']

train_encoded = pd.get_dummies(train_df, columns=onehot_cols, prefix=onehot_cols, drop_first=False)
val_encoded = pd.get_dummies(val_df, columns=onehot_cols, prefix=onehot_cols, drop_first=False)
test_encoded = pd.get_dummies(test_df, columns=onehot_cols, prefix=onehot_cols, drop_first=False)

train_cols = train_encoded.columns
val_encoded = val_encoded.reindex(columns=train_cols, fill_value=0)
test_encoded = test_encoded.reindex(columns=train_cols, fill_value=0)

train_df, val_df, test_df = train_encoded, val_encoded, test_encoded
print(f"Shape after one-hot encoding: {train_df.shape}")


Shape after one-hot encoding: (46026, 35)


**What we found:**

**Column alignment worked correctly with no errors, and the shapes confirm exactly what was expected.** `Shipping Mode` (4 values) + `Customer Segment` (3 values) + `Type` (4 values) = 11 total dummy columns replacing 3 original categorical columns, a net gain of 8 columns, matching the shift from the pre-encoding column count to 35 columns after encoding.

No `Customer Segment` or `Type` value turned out to be missing from any split (all three categoricals are low-cardinality enough, and the chronological split preserved full coverage here too, consistent with what Notebook 03 found for `Order Country`). The `reindex` safety net for handling a potentially missing category never had to activate, but it's confirmed correctly in place for robustness if this pipeline is ever re-run on different data (e.g. a future batch of new orders where a rare `Type` value might genuinely be missing from one split).


## 2. Frequency encode Category Name (fit on training data only, same leakage rule as Section 3's target encoding)

In [3]:
category_freq_map = train_df['Category Name'].value_counts(normalize=True)
global_freq_fallback = category_freq_map.min()  

train_df['category_frequency'] = train_df['Category Name'].map(category_freq_map).fillna(global_freq_fallback)
val_df['category_frequency'] = val_df['Category Name'].map(category_freq_map).fillna(global_freq_fallback)
test_df['category_frequency'] = test_df['Category Name'].map(category_freq_map).fillna(global_freq_fallback)

print(train_df[['Category Name', 'category_frequency']].head())


      Category Name  category_frequency
0  Camping & Hiking            0.158106
1    Men's Footwear            0.092926
2       Accessories            0.026898
3  Camping & Hiking            0.158106
4  Camping & Hiking            0.158106


## 3. Save

In [4]:
train_df.to_csv(TRAIN_OUT, index=False)
val_df.to_csv(VAL_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)
print("Saved step 4 outputs.")


Saved step 4 outputs.
